# TrustSafe-RAG | Gerador oficial v0.2 (45 consultas)

**Tarefa 1B.** Este notebook reconstrói o conjunto oficial de consultas de saúde mental a partir de um arquivo fonte mestre versionado. **Não** gera rótulos novos, não consulta serviços externos e não sobrescreve silenciosamente arquivos existentes.

O arquivo legado `consultas_v01.csv` é preservado. Os rótulos desta versão continuam **pendentes de revisão humana**.

## 1. Configuração de pastas

No Google Colab, extraia o ZIP completo em `/content` ou no Drive e configure `os.environ["TRUSTSAFE_PROJECT_ROOT"]` antes de executar. Não faça upload separado do CSV.

In [1]:
# Execute a partir da raiz do ZIP extraido (ou defina TRUSTSAFE_PROJECT_ROOT).
from pathlib import Path
import os, sys
candidatos = []
if os.environ.get("TRUSTSAFE_PROJECT_ROOT"):
    candidatos.append(Path(os.environ["TRUSTSAFE_PROJECT_ROOT"]).expanduser())
candidatos.extend([Path.cwd(), *Path.cwd().parents,
    Path("/content/TrustSafe-RAG-corrigido"),
    Path("/content/TrustSafe-RAG-main"),
    Path("/content/drive/MyDrive/TrustSafe-RAG-corrigido"),
    Path("/content/drive/MyDrive/TrustSafe-RAG")])
RAIZ = next((p.resolve() for p in candidatos
    if (p / "data/source/consultas_mental_v02_master.csv").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Extraia o ZIP inteiro e defina TRUSTSAFE_PROJECT_ROOT para a pasta do projeto. Nenhum dataset alternativo sera criado.")
sys.path.insert(0, str(RAIZ))
print("Projeto:", RAIZ)


Projeto: /mnt/data/_trustsafe_work/TrustSafe-RAG-corrigido


## 2. Geração e conferência do CSV v0.2

Se o CSV oficial já existir, ele é comparado à fonte mestre. Se estiver ausente, será escrito **uma única vez**; em seguida será relido e validado.

In [2]:
from scripts.gerar_v02 import main as gerar_ou_validar
arquivo = gerar_ou_validar()


CSV preexistente: verificando sem sobrescrever: /mnt/data/_trustsafe_work/TrustSafe-RAG-corrigido/data/raw/consultas_mental_v02.csv
VALIDADO: 45 consultas / 15 por classe / conteudo igual a fonte mestre
SHA256: 019855404fcea6850e79a5d6223f91758097be79d12130067753d23e6526bf42


## 3. Testes de integridade e auditoria

In [3]:
from src.trustsafe import load_v02, load_audit_v02
from IPython.display import display
df = load_v02(RAIZ)
auditoria = load_audit_v02(RAIZ, df)
print('Registros:', len(df), '| IDs únicos:', df['id'].nunique())
print('Distribuição:')
print(df['risco'].value_counts().to_string())
print('Linhas da auditoria de escopo:', len(auditoria))
display(df[['id','consulta','risco','revisao_necessaria']].head(8))


Registros: 45 | IDs únicos: 45
Distribuição:
risco
baixo       15
moderado    15
alto        15
Linhas da auditoria de escopo: 45


,id,consulta,risco,revisao_necessaria
0,Q001,Quais hábitos simples podem melhorar a qualida...,baixo,sim
1,Q002,Como posso criar uma rotina noturna mais tranq...,baixo,sim
2,Q003,Quais estratégias ajudam a lidar com o estress...,baixo,sim
3,Q004,Como organizar minhas tarefas para me sentir m...,baixo,sim
4,Q005,Quais são os benefícios da atividade física pa...,baixo,sim
5,Q006,Existe alguma técnica de respiração que ajude ...,baixo,sim
6,Q007,Como posso desenvolver hábitos de autocuidado ...,baixo,sim
7,Q008,Como manter contato com amigos pode contribuir...,baixo,sim


## 4. Interpretação

As verificações demonstram **integridade estrutural**, não acurácia de risco, consenso entre revisores nem validade clínica. A próxima etapa é o notebook `01_exploracao_consultas.ipynb`.